In [5]:
import os
import sys

print("=" * 60)
print("SETUP")
print("=" * 60)

# Detect environment
on_colab = 'google.colab' in sys.modules
print(f"Environment: {'Colab' if on_colab else 'Local'}")

if on_colab:
    base_dir = "/content"
    os.chdir(base_dir)

    # Clone repo if missing
    if not os.path.exists('trading-signals'):
        print("Cloning repo...")
        os.system('git clone https://github.com/tom-curtis/trading-signals.git')

    # Enter project directory
    os.chdir('trading-signals')

    print("Pulling latest...")
    os.system('git pull origin main -q')

    data_path = '/content/data'
else:
    os.chdir('.')  # local
    data_path = 'data'

project_dir = os.getcwd()
print(f"Project dir: {project_dir}")

# Ensure data dir exists
os.makedirs(data_path, exist_ok=True)

# Download Kaggle data if missing
csv_path = f'{data_path}/Combined_News_DJIA.csv'

if not os.path.exists(csv_path):
    print(f"Downloading Kaggle data to {data_path}...")
    os.system('pip install kaggle -q')
    os.system(f'kaggle datasets download -d aaron7sun/stocknews -p {data_path} --unzip -q')

# Add project to path
sys.path.insert(0, project_dir)

# Validate setup
print(f"✓ Data: {os.path.exists(csv_path)}")
print(f"✓ src.data_loader: ", end="")
try:
    from src.data_loader import load_kaggle_data
    print("OK")
except Exception as e:
    print(f"FAIL ({e})")

print("=" * 60)
print("Setup complete!\n")

SETUP
Environment: Colab
Pulling latest...
Project dir: /content/trading-signals
✓ Data: True
✓ src.data_loader: OK
Setup complete!



# Predicting Market Direction from News Headlines

This project investigates whether daily news headlines contain predictive signal for stock market movements. The task is formulated as a binary classification problem, where the objective is to predict whether the market will move up or down on the following trading day.

Two data sources are used: a dataset of news headlines and historical DJIA price data. These are combined to construct a dataset where each observation corresponds to a single trading day.

The approach consists of three components. First, headline text is used to train a sequence model for direction prediction. Second, price data is used to identify anomalous market conditions. Finally, these signals are combined to evaluate whether anomaly awareness improves decision-making.

## Data Preparation

The dataset consists of two sources: daily news headlines and historical DJIA market data. These are processed separately before being combined into a single modelling dataset.

In [ ]:
from src.data_loader import (
    load_headlines_csv,
    load_prices_csv,
    aggregate_headlines_by_day,
    add_price_features,
    merge_market_and_headlines,
    add_next_day_target,
    split_by_ratio
)

import pandas as pd

ImportError: cannot import name 'load_headlines_csv' from 'src.data_loader' (/content/trading-signals/src/data_loader.py)

Both datasets are loaded and dates are parsed into a consistent datetime format. Rows with missing or invalid values are removed. Headline text is cleaned by removing empty entries and normalising whitespace.

In [ ]:
headlines_df = load_headlines_csv(f"{data_path}/headlines.csv")
prices_df = load_prices_csv(f"{data_path}/djia.csv")

print(headlines_df.shape)
print(prices_df.shape)

headlines_df.head()

Headlines are aggregated by date so that each trading day corresponds to a single observation.

In [ ]:
daily_headlines_df = aggregate_headlines_by_day(headlines_df)

daily_headlines_df.head()

Additional features are computed from the market data, including daily return, intraday range, and changes relative to the previous day.

In [ ]:
prices_df = add_price_features(prices_df)

prices_df.head()

The headline and market datasets are merged on date, retaining only days where both sources are available.

In [ ]:
dataset_df = merge_market_and_headlines(prices_df, daily_headlines_df)

dataset_df.head()

A binary target is defined based on next-day price movement. A value of 1 indicates that the next trading day closes higher than the current day, while a value of 0 indicates that it does not. This means flat and negative next-day movements are grouped together as the non-positive class.

In [ ]:
dataset_df = add_next_day_target(dataset_df, drop_no_change_days=False)

dataset_df.head()

The dataset is split chronologically into training, validation, and test sets using a 70/15/15 ratio. This preserves temporal order while reserving later observations for model selection and final evaluation.

In [ ]:
train_df, val_df, test_df = split_by_ratio(
    dataset_df,
    train_ratio=0.70,
    val_ratio=0.15,
)

print(len(train_df), len(val_df), len(test_df))

print("\nTrain range:", train_df["Date"].min(), "→", train_df["Date"].max())
print("Val range:", val_df["Date"].min(), "→", val_df["Date"].max())
print("Test range:", test_df["Date"].min(), "→", test_df["Date"].max())

## Phase 1: Predicting Market Direction from Headlines

This phase evaluates whether daily news headlines alone contain predictive signal for next-day market direction. Only headline text and the target variable are retained for this phase.

In [ ]:
train_text_df = train_df[["text", "target"]].copy()
val_text_df = val_df[["text", "target"]].copy()
test_text_df = test_df[["text", "target"]].copy()

In [ ]:
X_train = train_text_df["text"].astype(str).values
X_val = val_text_df["text"].astype(str).values
X_test = test_text_df["text"].astype(str).values

y_train = train_text_df["target"].values
y_val = val_text_df["target"].values
y_test = test_text_df["target"].values

### Baseline

A majority-class baseline is used as a reference point for model performance.

In [7]:
from sklearn.metrics import accuracy_score

majority_class = train_text_df["target"].mode()[0]

val_pred = [majority_class] * len(val_text_df)
test_pred = [majority_class] * len(test_text_df)

print("Majority class:", majority_class)
print("Validation accuracy:", accuracy_score(val_text_df["target"], val_pred))
print("Test accuracy:", accuracy_score(test_text_df["target"], test_pred))

NameError: name 'train_text_df' is not defined

### Prepare inputs

In [8]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

X_train = train_text_df["text"].astype(str).values
X_val = val_text_df["text"].astype(str).values
X_test = test_text_df["text"].astype(str).values

y_train = train_text_df["target"].values
y_val = val_text_df["target"].values
y_test = test_text_df["target"].values

max_tokens = 10000
sequence_length = 200

vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length,
)

vectorizer.adapt(X_train)

NameError: name 'train_text_df' is not defined

The data is converted into batched TensorFlow datasets for training.

In [ ]:
batch_size = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

An LSTM model is used to capture sequential patterns in the headline text.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

embedding_dim = 64

model = Sequential([
    vectorizer,
    Embedding(input_dim=max_tokens, output_dim=embedding_dim, mask_zero=True),
    LSTM(64),
    Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
)